# fact_sales — Star Schema Fact Loader (PySpark)

In [0]:
# ============================================================
#  fact_sales  — Star schema fact loader (PySpark)
# ============================================================
#  Joins silver.cleanedSales to all conformed dimension surrogate
#  keys, computes derived measures (profit, margin), writes
#  incrementally via Delta append with watermark.
# ============================================================
from pyspark.sql import functions as F

dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")

silver_sales = f"saleslake_{env}.silver_{env}.cleanedSales"
gold_fact    = f"saleslake_{env}.gold_{env}.fact_sales"
gold_g       = f"saleslake_{env}.gold_{env}"

# --- last watermark already loaded ------------------------------
watermark = (spark.sql(
    f"SELECT COALESCE(MAX(load_ts), TO_TIMESTAMP('1990-01-01')) AS w "
    f"FROM {gold_fact}").first()["w"])
print(f"Loading sales with ingest_ts > {watermark}")

df_s   = spark.table(silver_sales).filter(F.col("ingest_ts") > F.lit(watermark))
d_cust = spark.table(f"{gold_g}.refinedCustomer").filter("is_active = 'Y'") \
              .select("customer_sk", "customer_id")
d_prod = spark.table(f"{gold_g}.dim_product").filter("is_active = 'Y'") \
              .select("product_sk", "product_id")
d_stor = spark.table(f"{gold_g}.dim_store").filter("is_active = 'Y'") \
              .select("store_sk", "store_id")
d_reg  = spark.table(f"{gold_g}.dim_region") \
              .select("region_sk", "region_id")
d_disc = spark.table(f"{gold_g}.dim_discount").filter("is_active = 'Y'") \
              .select("discount_sk", "discount_code")
d_date = spark.table(f"{gold_g}.dim_date").select("date_sk", "full_date")

fact = (df_s
        .join(d_cust, "customer_id", "left")
        .join(d_prod, "product_id",  "left")
        .join(d_stor, "store_id",    "left")
        .join(d_reg,  "region_id",   "left")
        .join(d_disc, "discount_code", "left")
        .join(d_date.alias("dd"), F.col("sale_date") == F.col("dd.full_date"), "left")
        .withColumn("profit_amount",
              (F.col("net_amount") - F.col("cost_amount")).cast("decimal(18,2)"))
        .withColumn("profit_margin_pct",
              F.when(F.col("net_amount") > 0,
                     F.round(F.col("profit_amount") / F.col("net_amount") * 100, 4).cast("decimal(8,4)"))
               .otherwise(F.lit(0).cast("decimal(8,4)")))
        .select(
            F.col("sale_id"),
            F.col("date_sk"),
            F.col("customer_sk"), F.col("product_sk"),
            F.col("store_sk"),    F.col("region_sk"),
            F.col("discount_sk"),
            F.col("sale_date"),
            F.col("quantity"),
            F.col("unit_price"),
            F.col("gross_amount"),
            F.col("discount_amount"),
            F.col("net_amount"),
            F.col("cost_amount"),
            F.col("profit_amount"),
            F.col("profit_margin_pct"),
            F.col("payment_method"),
            F.col("channel"),
            F.current_timestamp().alias("load_ts"),
        ))

fact.write.format("delta").mode("append").saveAsTable(gold_fact)
print(f"fact_sales loaded: +{fact.count()} rows")
spark.sql(f"SELECT COUNT(*) AS total_fact_rows FROM {gold_fact}").display()